# Problema 1: Análise de Custos de Serviços em Nuvem

Projeto Métodos Numéricos 2ºGQ. As funções vêm do pacote `custos_nuvem` (instale com `pip install -e .` na raiz do repositório).

In [1]:
import numpy as np
import pandas as pd
from custos_nuvem import (
    carregar_csv, vetor_previsao, montar_sistema, custos_unitarios,
    verificar_totais, custo_previsto, melhor_reducao,
    eliminacao_gauss, resolver_lu,
)

## Dados

Ajuste os nomes das colunas conforme os CSVs (e `sep`/`decimal` se necessário).

In [2]:
consumo = carregar_csv('../data/consumo_servicos.csv')
previsao = carregar_csv('../data/previsao.csv')
display(consumo)
display(previsao)

,mes,cpu_horas,armazenamento_tb,banco_horas,transferencia_gb,funcoes_mil,custo_total
0,1,100.0,20.0,80.0,200.0,150.0,1255.0
1,2,120.0,25.0,90.0,180.0,200.0,1479.0
2,3,150.0,30.0,100.0,250.0,180.0,1695.0
3,4,130.0,35.0,120.0,220.0,250.0,1881.0
4,5,180.0,40.0,110.0,300.0,220.0,2060.0


,mes,cpu_horas,armazenamento_tb,banco_horas,transferencia_gb,funcoes_mil
0,6,200.0,45.0,130.0,320.0,280.0


In [3]:
# Colunas dos CSVs do enunciado
SERVICOS = ['cpu_horas', 'armazenamento_tb', 'banco_horas', 'transferencia_gb', 'funcoes_mil']
COLUNA_TOTAL = 'custo_total'

## (a) Custo unitário de cada serviço

Sistema linear 5×5 `A·x = b`: `A` = quantidades consumidas, `b` = total pago por mês.

In [4]:
A, b = montar_sistema(consumo, SERVICOS, COLUNA_TOTAL)
x = custos_unitarios(A, b)              # eliminação de Gauss com pivoteamento
x_lu = resolver_lu(A, b)                # conferência com LU
pd.DataFrame({'gauss': x, 'lu': x_lu}, index=SERVICOS)

,gauss,lu
cpu_horas,2.5,2.5
armazenamento_tb,15.0,15.0
banco_horas,4.0,4.0
transferencia_gb,0.8,0.8
funcoes_mil,1.5,1.5


## (b) Verificação dos totais de faturamento

In [5]:
verificar_totais(A, x, b)

,mes,total_calculado,total_original,erro_absoluto,erro_relativo
0,1,1255.0,1255.0,0.0,0.0
1,2,1479.0,1479.0,0.0,0.0
2,3,1695.0,1695.0,0.0,0.0
3,4,1881.0,1881.0,0.0,0.0
4,5,2060.0,2060.0,0.0,0.0


**Interpretação:** _(Guilherme: comentar erros, arredondamento e condicionamento.)_

## (c) Custo total estimado do próximo mês

O custo previsto é a soma, sobre os cinco serviços, de (quantidade prevista × custo unitário). Os custos unitários vêm do item (a).

In [6]:
q = vetor_previsao(previsao, SERVICOS)

tabela_c = pd.DataFrame(
    {"quantidade_prevista": q, "custo_unitario": x, "custo": q * x},
    index=SERVICOS,
)
display(tabela_c.round(2))
print(f"Custo total estimado do mês 6: {custo_previsto(q, x):.2f}")

,quantidade_prevista,custo_unitario,custo
cpu_horas,200.0,2.5,500.0
armazenamento_tb,45.0,15.0,675.0
banco_horas,130.0,4.0,520.0
transferencia_gb,320.0,0.8,256.0
funcoes_mil,280.0,1.5,420.0


Custo total estimado do mês 6: 2371.00


**Interpretação (c):** o custo total estimado para o mês 6 é **2371,00**. O armazenamento é o maior item (675,00, cerca de 28,5% do total), seguido por banco de dados (520,00) e CPU (500,00). Transferência de dados, apesar de ter a maior quantidade (320 GB), pesa pouco (256,00), porque seu custo unitário é baixo (0,80).

Essa estimativa vale se os preços unitários se mantiverem iguais aos dos cinco meses analisados e se a previsão de consumo (`previsao.csv`) se confirmar. O modelo converte consumo em custo, mas não prevê o consumo em si.

## (d) Redução de 20% em um serviço

Reduzir uma fração f do uso do serviço *i* economiza f × (quantidade_i × custo_i). O critério de escolha é, portanto, o **gasto absoluto** de cada serviço no mês, e não o seu preço unitário.

In [7]:
r = melhor_reducao(q, x, SERVICOS, reducao=0.20, meses=12)

print('Serviço escolhido:', r['servico'])
print(f"Economia mensal: {r['economia_mensal']:.2f}")
print(f"Novo custo total: {r['novo_total']:.2f}")
print(f"Economia em 12 meses: {r['economia_acumulada']:.2f}")

r['cenarios'].round(2)

Serviço escolhido: armazenamento_tb
Economia mensal: 135.00
Novo custo total: 2236.00
Economia em 12 meses: 1620.00


,custo_atual_servico,economia_mensal,novo_total,economia_acumulada
armazenamento_tb,675.0,135.0,2236.0,1620.0
banco_horas,520.0,104.0,2267.0,1248.0
cpu_horas,500.0,100.0,2271.0,1200.0
funcoes_mil,420.0,84.0,2287.0,1008.0
transferencia_gb,256.0,51.2,2319.8,614.4


**Verificação independente:** em vez de subtrair a economia, reduzimos 20% da *quantidade* do serviço escolhido e recalculamos o custo total do zero. Os dois caminhos devem coincidir.

In [8]:
q_reduzida = q.copy()
q_reduzida[SERVICOS.index(r['servico'])] *= 0.8

novo_total_recalculado = custo_previsto(q_reduzida, x)
print(f"Novo total (subtraindo a economia): {r['novo_total']:.2f}")
print(f"Novo total (recalculando do zero):  {novo_total_recalculado:.2f}")
assert abs(novo_total_recalculado - r['novo_total']) < 1e-9

Novo total (subtraindo a economia): 2236.00
Novo total (recalculando do zero):  2236.00


**Interpretação (d):** o serviço que proporciona a maior economia é o **armazenamento**: reduzir 20% do seu uso economiza **135,00 por mês**, levando o custo total de 2371,00 para **2236,00**. Mantido o mesmo padrão de uso, a economia acumulada em 12 meses é de **1620,00** (12 × 135,00).

O armazenamento vence porque tem o maior gasto absoluto (675,00) e, ao mesmo tempo, o maior preço unitário (15,00). Os dois fatores não coincidem sempre: a CPU (500,00) fica atrás do banco de dados (520,00) mesmo tendo preço unitário menor, e a transferência é a pior opção (51,20 por mês) por ter preço unitário baixo, apesar da grande quantidade.

A projeção de 12 meses supõe consumo constante, como pede o enunciado. Como o consumo vem crescendo ao longo dos meses (a CPU foi de 100 para 200 horas), a economia real tenderia a ser maior do que 1620,00.

## Conclusões e limitações

_Preencher._